<a href="https://colab.research.google.com/github/microsoft/qlib/blob/main/examples/my_workflow_visual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Copyright (c) Microsoft Corporation.
#  Licensed under the MIT License.

In [43]:
import sys, site
from pathlib import Path

################################# NOTE #################################
#  Please be aware that if colab installs the latest numpy and pyqlib  #
#  in this cell, users should RESTART the runtime in order to run the  #
#  following cells successfully.                                       #
########################################################################

try:
    import qlib
except ImportError:
    # install qlib
    ! pip install --upgrade numpy
    ! pip install pyqlib
    if "google.colab" in sys.modules:
        ! pip install pyyaml==5.4.1
    # reload
    site.main()

scripts_dir = Path.cwd().parent.joinpath("scripts")
if not scripts_dir.joinpath("get_data.py").exists():
    # download get_data.py script
    scripts_dir = Path("~/tmp/qlib_code/scripts").expanduser().resolve()
    scripts_dir.mkdir(parents=True, exist_ok=True)
    import requests

    with requests.get("https://raw.githubusercontent.com/microsoft/qlib/main/scripts/get_data.py", timeout=10) as resp:
        with open(scripts_dir.joinpath("get_data.py"), "wb") as fp:
            fp.write(resp.content)

In [44]:
import qlib
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from qlib.constant import REG_CN
from qlib.utils import exists_qlib_data, init_instance_by_config
from qlib.workflow import R
from qlib.workflow.record_temp import SignalRecord, PortAnaRecord
from qlib.utils import flatten_dict
from qlib.data import D
from qlib.backtest import backtest
from qlib.backtest.executor import SimulatorExecutor
from qlib.contrib.strategy import TopkDropoutStrategy
from qlib.contrib.evaluate import risk_analysis
from qlib.utils.time import Freq

# matplotlib 中文字体
import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

# 动态计算数据截止日期：三点之后用今天，否则用昨天
now = datetime.now()
today = now.date()
if now.hour >= 15:
    cutoff_date = today
else:
    cutoff_date = today - timedelta(days=1)
cutoff_str = cutoff_date.strftime("%Y-%m-%d")

print(f"今天: {today}")
print(f"当前时间: {now.strftime('%H:%M:%S')}")
print(f"数据截止日期: {cutoff_str}")

今天: 2026-05-07
当前时间: 08:32:27
数据截止日期: 2026-05-06


In [45]:
# Download Qlib data (will fetch latest available data)
provider_uri = "~/.qlib/qlib_data/cn_data"
if not exists_qlib_data(provider_uri):
    print(f"Qlib data is not found in {provider_uri}")
    sys.path.append(str(scripts_dir))
    from get_data import GetData

    GetData().qlib_data(target_dir=provider_uri, region=REG_CN)

qlib.init(provider_uri=provider_uri, region=REG_CN)

# Check available date range in the data
instruments = D.instruments(market="csi300")
calendar = D.calendar(start_time=None, end_time=None)
print(f"Data calendar range: {calendar[0]} ~ {calendar[-1]}")
print(f"Number of trading days available: {len(calendar)}")

[52360:MainThread](2026-05-07 08:32:28,691) INFO - qlib.Initialization - [config.py:453] - default_conf: client.
[52360:MainThread](2026-05-07 08:32:28,695) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.
[52360:MainThread](2026-05-07 08:32:28,697) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': WindowsPath('C:/Users/chen/.qlib/qlib_data/cn_data')}


Data calendar range: 2000-01-04 00:00:00 ~ 2026-04-30 00:00:00
Number of trading days available: 6378


In [46]:
# Market & benchmark config
market = "csi300"
benchmark = "SH000300"

# Dynamically compute train/valid/test split based on cutoff date
train_start = (cutoff_date - timedelta(days=365 * 10)).strftime("%Y-%m-%d")
train_end = (cutoff_date - timedelta(days=365 * 2)).strftime("%Y-%m-%d")
valid_start = train_end
valid_end = (cutoff_date - timedelta(days=365 * 1)).strftime("%Y-%m-%d")
test_start = valid_end
test_end = cutoff_str

print(f"Train period: {train_start} ~ {train_end}")
print(f"Valid period: {valid_start} ~ {valid_end}")
print(f"Test  period: {test_start} ~ {test_end}")

Train period: 2016-05-08 ~ 2024-05-06
Valid period: 2024-05-06 ~ 2025-05-06
Test  period: 2025-05-06 ~ 2026-05-06


## Step 1a: Train LGBModel

In [47]:
###################################
# Train LGBModel
###################################
data_handler_config = {
    "start_time": train_start,
    "end_time": test_end,
    "fit_start_time": train_start,
    "fit_end_time": train_end,
    "instruments": market,
}

lgb_task = {
    "model": {
        "class": "LGBModel",
        "module_path": "qlib.contrib.model.gbdt",
        "kwargs": {
            "loss": "mse",
            "colsample_bytree": 0.8879,
            "learning_rate": 0.0421,
            "subsample": 0.8789,
            "lambda_l1": 205.6999,
            "lambda_l2": 580.9768,
            "max_depth": 8,
            "num_leaves": 210,
            "num_threads": 20,
        },
    },
    "dataset": {
        "class": "DatasetH",
        "module_path": "qlib.data.dataset",
        "kwargs": {
            "handler": {
                "class": "Alpha158",
                "module_path": "qlib.contrib.data.handler",
                "kwargs": data_handler_config,
            },
            "segments": {
                "train": (train_start, train_end),
                "valid": (valid_start, valid_end),
                "test": (test_start, test_end),
            },
        },
    },
}

print("Initializing LGBModel...")
lgb_model = init_instance_by_config(lgb_task["model"])
lgb_dataset = init_instance_by_config(lgb_task["dataset"])

print("Training LGBModel...")
with R.start(experiment_name="train_lgb"):
    R.log_params(**flatten_dict(lgb_task))
    lgb_model.fit(lgb_dataset)
    R.save_objects(trained_model=lgb_model)
    lgb_rid = R.get_recorder().id

print(f"LGBModel training completed! Recorder ID: {lgb_rid}")

Initializing LGBModel...


[52360:MainThread](2026-05-07 08:33:18,266) INFO - qlib.timer - [log.py:127] - Time cost: 49.520s | Loading data Done
[52360:MainThread](2026-05-07 08:33:18,938) INFO - qlib.timer - [log.py:127] - Time cost: 0.143s | DropnaLabel Done
[52360:MainThread](2026-05-07 08:33:20,182) INFO - qlib.timer - [log.py:127] - Time cost: 1.243s | CSZScoreNorm Done
[52360:MainThread](2026-05-07 08:33:20,219) INFO - qlib.timer - [log.py:127] - Time cost: 1.950s | fit & process data Done
[52360:MainThread](2026-05-07 08:33:20,220) INFO - qlib.timer - [log.py:127] - Time cost: 51.475s | Init data Done


Training LGBModel...


[52360:MainThread](2026-05-07 08:33:20,229) WARNING - qlib.workflow - [expm.py:230] - No valid experiment found. Create a new experiment with name train_lgb.
[52360:MainThread](2026-05-07 08:33:20,238) INFO - qlib.workflow - [exp.py:258] - Experiment 634576823384325991 starts running ...
[52360:MainThread](2026-05-07 08:33:20,308) INFO - qlib.workflow - [recorder.py:345] - Recorder 026d6f6842aa4b1a97ec8614364dd29e starts running under Experiment 634576823384325991 ...


Training until validation scores don't improve for 50 rounds
[20]	train's l2: 0.993477	valid's l2: 0.995866
[40]	train's l2: 0.991483	valid's l2: 0.995602
[60]	train's l2: 0.989818	valid's l2: 0.995526
[80]	train's l2: 0.988252	valid's l2: 0.995514
[100]	train's l2: 0.986804	valid's l2: 0.995578
[120]	train's l2: 0.985388	valid's l2: 0.995657
Early stopping, best iteration is:
[76]	train's l2: 0.988573	valid's l2: 0.995484


[52360:MainThread](2026-05-07 08:33:29,295) INFO - qlib.timer - [log.py:127] - Time cost: 0.230s | waiting `async_log` Done


LGBModel training completed! Recorder ID: 026d6f6842aa4b1a97ec8614364dd29e


## Step 1b: LGBModel Predictions

In [48]:
###################################
# LGBModel predictions
###################################
with R.start(experiment_name="prediction_lgb"):
    recorder = R.get_recorder(recorder_id=lgb_rid, experiment_name="train_lgb")
    model = recorder.load_object("trained_model")

    recorder = R.get_recorder()
    sr = SignalRecord(model, lgb_dataset, recorder)
    sr.generate()
    lgb_pred_rid = recorder.id

recorder = R.get_recorder(recorder_id=lgb_pred_rid, experiment_name="prediction_lgb")
lgb_pred_df = recorder.load_object("pred.pkl")
print(f"LGBModel predictions shape: {lgb_pred_df.shape}")
print(f"Prediction date range: {lgb_pred_df.index.get_level_values('datetime').min()} ~ {lgb_pred_df.index.get_level_values('datetime').max()}")
lgb_pred_df.head(5)

[52360:MainThread](2026-05-07 08:33:29,334) WARNING - qlib.workflow - [expm.py:230] - No valid experiment found. Create a new experiment with name prediction_lgb.
[52360:MainThread](2026-05-07 08:33:29,347) INFO - qlib.workflow - [exp.py:258] - Experiment 167508167080253451 starts running ...
[52360:MainThread](2026-05-07 08:33:29,402) INFO - qlib.workflow - [recorder.py:345] - Recorder d33af8f0f9b343f8be1e9308bececb63 starts running under Experiment 167508167080253451 ...


[52360:MainThread](2026-05-07 08:33:29,964) INFO - qlib.workflow - [record_temp.py:197] - Signal record 'pred.pkl' has been saved as the artifact of the Experiment 167508167080253451


'The following are prediction results of the LGBModel model.'
                          score
datetime   instrument          
2025-05-06 SH600000    0.025682
           SH600009    0.010076
           SH600010    0.043905
           SH600011    0.032705
           SH600015   -0.005264


[52360:MainThread](2026-05-07 08:33:29,990) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


LGBModel predictions shape: (72459, 1)
Prediction date range: 2025-05-06 00:00:00 ~ 2026-04-30 00:00:00


score
datetime   instrument          
2025-05-06 SH600000    0.025682
           SH600009    0.010076
           SH600010    0.043905
           SH600011    0.032705
           SH600015   -0.005264

## Step 1c: Train ALSTM

In [ ]:
###################################
# Train ALSTM (Attention LSTM)
###################################
alstm_task = {
    "model": {
        "class": "ALSTM",
        "module_path": "qlib.contrib.model.pytorch_alstm",
        "kwargs": {
            "d_feat": 6,
            "hidden_size": 128,
            "num_layers": 2,
            "dropout": 0.1,
            "n_epochs": 100,
            "lr": 0.001,
            "early_stop": 10,
            "batch_size": 2000,
            "metric": "loss",
            "loss": "mse",
            "GPU": 0,
        },
    },
    "dataset": {
        "class": "DatasetH",
        "module_path": "qlib.data.dataset",
        "kwargs": {
            "handler": {
                "class": "Alpha360",
                "module_path": "qlib.contrib.data.handler",
                "kwargs": data_handler_config,
            },
            "segments": {
                "train": (train_start, train_end),
                "valid": (valid_start, valid_end),
                "test": (test_start, test_end),
            },
        },
    },
}

print("Initializing ALSTM...")
alstm_model = init_instance_by_config(alstm_task["model"])
alstm_dataset = init_instance_by_config(alstm_task["dataset"])

print("Training ALSTM (this may take several minutes)...")
with R.start(experiment_name="train_alstm"):
    R.log_params(**flatten_dict(alstm_task))
    alstm_model.fit(alstm_dataset)
    R.save_objects(trained_model=alstm_model)
    alstm_rid = R.get_recorder().id

print(f"ALSTM training completed! Recorder ID: {alstm_rid}")

Initializing ALSTM...


[52360:MainThread](2026-05-07 08:49:12,091) INFO - qlib.ALSTM - [pytorch_alstm.py:59] - ALSTM pytorch version...
[52360:MainThread](2026-05-07 08:49:12,093) INFO - qlib.ALSTM - [pytorch_alstm.py:76] - ALSTM parameters setting:
d_feat : 6
hidden_size : 128
num_layers : 2
dropout : 0.1
n_epochs : 100
lr : 0.001
metric : loss
batch_size : 2000
early_stop : 10
optimizer : adam
loss_type : mse
device : cpu
use_GPU : False
seed : None
[52360:MainThread](2026-05-07 08:49:12,095) INFO - qlib.ALSTM - [pytorch_alstm.py:119] - model:
ALSTMModel(
  (net): Sequential(
    (fc_in): Linear(in_features=6, out_features=128, bias=True)
    (act): Tanh()
  )
  (rnn): GRU(128, 128, num_layers=2, batch_first=True, dropout=0.1)
  (fc_out): Linear(in_features=256, out_features=1, bias=True)
  (att_net): Sequential(
    (att_fc_in): Linear(in_features=128, out_features=64, bias=True)
    (att_dropout): Dropout(p=0.1, inplace=False)
    (att_act): Tanh()
    (att_fc_out): Linear(in_features=64, out_features=1,

Training ALSTM (this may take several minutes)...


[52360:MainThread](2026-05-07 08:52:08,573) INFO - qlib.workflow - [exp.py:258] - Experiment 689530161949919692 starts running ...
[52360:MainThread](2026-05-07 08:52:08,620) INFO - qlib.workflow - [recorder.py:345] - Recorder 3c3b1def72834ab88fb4670272f91535 starts running under Experiment 689530161949919692 ...
[52360:MainThread](2026-05-07 08:52:09,948) INFO - qlib.ALSTM - [pytorch_alstm.py:235] - training...
[52360:MainThread](2026-05-07 08:52:09,950) INFO - qlib.ALSTM - [pytorch_alstm.py:239] - Epoch0:
[52360:MainThread](2026-05-07 08:52:09,952) INFO - qlib.ALSTM - [pytorch_alstm.py:240] - training...
[52360:MainThread](2026-05-07 08:56:22,311) INFO - qlib.ALSTM - [pytorch_alstm.py:242] - evaluating...
[52360:MainThread](2026-05-07 08:57:44,100) INFO - qlib.ALSTM - [pytorch_alstm.py:245] - train -0.996009, valid -0.996626
[52360:MainThread](2026-05-07 08:57:44,107) INFO - qlib.ALSTM - [pytorch_alstm.py:239] - Epoch1:
[52360:MainThread](2026-05-07 08:57:44,109) INFO - qlib.ALSTM - 

## Step 1d: ALSTM Predictions

In [ ]:
###################################
# ALSTM predictions
###################################
with R.start(experiment_name="prediction_alstm"):
    recorder = R.get_recorder(recorder_id=alstm_rid, experiment_name="train_alstm")
    model = recorder.load_object("trained_model")

    recorder = R.get_recorder()
    sr = SignalRecord(model, alstm_dataset, recorder)
    sr.generate()
    alstm_pred_rid = recorder.id

recorder = R.get_recorder(recorder_id=alstm_pred_rid, experiment_name="prediction_alstm")
alstm_pred_df = recorder.load_object("pred.pkl")
print(f"ALSTM predictions shape: {alstm_pred_df.shape}")
print(f"Prediction date range: {alstm_pred_df.index.get_level_values('datetime').min()} ~ {alstm_pred_df.index.get_level_values('datetime').max()}")
alstm_pred_df.head(5)

## Step 2.5: Backtest

In [ ]:
###################################
# Run backtest — LGBModel
###################################
STRATEGY_CONFIG = {
    "topk": 50,
    "n_drop": 5,
    "signal": lgb_pred_df,
}
EXECUTOR_CONFIG = {
    "time_per_step": "day",
    "generate_portfolio_metrics": True,
}
backtest_config = {
    "start_time": test_start,
    "end_time": test_end,
    "account": 100000000,
    "benchmark": benchmark,
    "exchange_kwargs": {
        "freq": "day",
        "limit_threshold": 0.095,
        "deal_price": "close",
        "open_cost": 0.0005,
        "close_cost": 0.0015,
        "min_cost": 5,
    },
}

strategy_obj = TopkDropoutStrategy(**STRATEGY_CONFIG)
executor_obj = SimulatorExecutor(**EXECUTOR_CONFIG)
portfolio_metric_dict, indicator_dict = backtest(
    executor=executor_obj, strategy=strategy_obj, **backtest_config
)

analysis_freq = f"{Freq.parse('day')[0]}{Freq.parse('day')[1]}"
lgb_report_normal_df, lgb_positions = portfolio_metric_dict.get(analysis_freq)

print("=" * 60)
print(f"  LGBModel 回测结果 ({test_start} ~ {test_end})")
print("=" * 60)
print(f"回测天数: {len(lgb_report_normal_df)}")
print("\n===== LGBModel 策略收益（含成本） =====")
display(risk_analysis(lgb_report_normal_df["return"] - lgb_report_normal_df["bench"] - lgb_report_normal_df["cost"]))

In [ ]:
###################################
# Run backtest — ALSTM
###################################
STRATEGY_CONFIG["signal"] = alstm_pred_df
strategy_obj = TopkDropoutStrategy(**STRATEGY_CONFIG)
portfolio_metric_dict, indicator_dict = backtest(
    executor=executor_obj, strategy=strategy_obj, **backtest_config
)

alstm_report_normal_df, alstm_positions = portfolio_metric_dict.get(analysis_freq)

print("=" * 60)
print(f"  ALSTM 回测结果 ({test_start} ~ {test_end})")
print("=" * 60)
print(f"回测天数: {len(alstm_report_normal_df)}")
print("\n===== ALSTM 策略收益（含成本） =====")
display(risk_analysis(alstm_report_normal_df["return"] - alstm_report_normal_df["bench"] - alstm_report_normal_df["cost"]))

# ── 对比汇总 ──
lgb_return = (1 + lgb_report_normal_df["return"] - lgb_report_normal_df["cost"]).cumprod().values[-1] - 1
alstm_return = (1 + alstm_report_normal_df["return"] - alstm_report_normal_df["cost"]).cumprod().values[-1] - 1
bench_return = (1 + alstm_report_normal_df["bench"]).cumprod().values[-1] - 1

print("\n" + "=" * 60)
print(f"  双模型对比汇总")
print("=" * 60)
print(f"  LGBModel  累计收益: {lgb_return:.4%}")
print(f"  ALSTM     累计收益: {alstm_return:.4%}")
print(f"  基准      累计收益: {bench_return:.4%}")
print(f"  LGBModel  超额收益: {lgb_return - bench_return:.4%}")
print(f"  ALSTM     超额收益: {alstm_return - bench_return:.4%}")
if alstm_return > lgb_return:
    print(f"\n  >>> ALSTM 表现更优，超额领先 {alstm_return - lgb_return:.4%}")
else:
    print(f"\n  >>> LGBModel 表现更优，超额领先 {lgb_return - alstm_return:.4%}")

## Step 2.6: 收益率曲线 — 双模型对比

In [ ]:
###################################
# 绘制双模型收益率曲线对比
###################################
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── 计算各模型累计收益 ──
def calc_cum_returns(report_df):
    strategy_net = report_df["return"] - report_df["cost"]
    return (1 + strategy_net).cumprod()

cum_lgb = calc_cum_returns(lgb_report_normal_df)
cum_alstm = calc_cum_returns(alstm_report_normal_df)
cum_bench = (1 + alstm_report_normal_df["bench"].fillna(0)).cumprod()

final_lgb = cum_lgb.values[-1] - 1
final_alstm = cum_alstm.values[-1] - 1
final_bench = cum_bench.values[-1] - 1

fig, axes = plt.subplots(2, 1, figsize=(16, 12))

# ── 上图: LGBModel vs ALSTM vs 基准 累计收益 ──
ax1 = axes[0]
ax1.plot(cum_lgb.index, cum_lgb.values, label="LGBModel（含成本）", color="#2980b9", linewidth=1.8)
ax1.plot(cum_alstm.index, cum_alstm.values, label="ALSTM（含成本）", color="#e67e22", linewidth=1.8)
ax1.plot(cum_bench.index, cum_bench.values, label=f"基准 ({benchmark})", color="#95a5a6", linewidth=1.5, linestyle="--")
ax1.set_title(f"双模型累计收益对比 ({test_start} ~ {test_end})", fontsize=14, fontweight="bold")
ax1.set_ylabel("累计收益", fontsize=11)
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax1.legend(loc="upper left", fontsize=9)
ax1.grid(True, alpha=0.3)

# 标注最终收益
ax1.text(0.02, 0.95,
    f"LGBModel  最终收益: {final_lgb:.2%}\n"
    f"ALSTM     最终收益: {final_alstm:.2%}\n"
    f"基准      最终收益: {final_bench:.2%}",
    transform=ax1.transAxes, fontsize=10, verticalalignment="top",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

# ── 下图: 超额收益对比 ──
ax2 = axes[1]
lgb_excess = cum_lgb.values.flatten() - cum_bench.values.flatten()
alstm_excess = cum_alstm.values.flatten() - cum_bench.values.flatten()
ax2.plot(cum_lgb.index, lgb_excess, label="LGBModel 超额收益", color="#2980b9", linewidth=1.5)
ax2.plot(cum_alstm.index, alstm_excess, label="ALSTM 超额收益", color="#e67e22", linewidth=1.5)
ax2.axhline(y=0, color="black", linewidth=0.5, linestyle="--")
ax2.fill_between(cum_lgb.index, lgb_excess, 0, alpha=0.08, color="#2980b9")
ax2.fill_between(cum_alstm.index, alstm_excess, 0, alpha=0.08, color="#e67e22")
ax2.set_title("双模型超额收益对比", fontsize=14, fontweight="bold")
ax2.set_ylabel("超额收益", fontsize=11)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax2.legend(loc="upper left", fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 3: Today's Stock Recommendations (Best Model)

In [ ]:
###################################
# Get today's stock recommendations
# using the best-performing model
###################################

# Auto-select best model based on backtest returns
lgb_final = (1 + lgb_report_normal_df["return"] - lgb_report_normal_df["cost"]).cumprod().values[-1] - 1
alstm_final = (1 + alstm_report_normal_df["return"] - alstm_report_normal_df["cost"]).cumprod().values[-1] - 1

if alstm_final > lgb_final:
    best_model_name = "ALSTM"
    pred_df = alstm_pred_df
else:
    best_model_name = "LGBModel"
    pred_df = lgb_pred_df

print(f"Best model: {best_model_name}")
print(f"  LGBModel  final return: {lgb_final:.4%}")
print(f"  ALSTM     final return: {alstm_final:.4%}")

# Get the latest trading day's predictions
latest_date = pred_df.index.get_level_values("datetime").max()
latest_pred = pred_df.loc[pred_df.index.get_level_values("datetime") == latest_date]
latest_pred = latest_pred.droplevel("datetime")

print(f"\nLatest trading day with predictions: {latest_date}")

# Rank stocks by prediction score (descending) and pick top 20
top_n = 20
top_stocks = latest_pred.sort_values("score", ascending=False).head(top_n)

print(f"\n{'='*60}")
print(f"  Today's ({today}) Top {top_n} Recommended Stocks")
print(f"  Model: {best_model_name} | Data as of {latest_date}")
print(f"{'='*60}\n")

for rank, (stock, row) in enumerate(top_stocks.iterrows(), 1):
    print(f"  #{rank:<4} {stock:<12} Score: {row['score']:.6f}")

print(f"\n{'='*60}")

In [ ]:
###################################
# Detailed recommendation summary
###################################

# Create a clean recommendation table
recommendations = top_stocks.copy()
recommendations.index.name = "Stock"
recommendations.columns = ["Prediction Score"]
recommendations.insert(0, "Rank", range(1, len(recommendations) + 1))

# Style the output
print(f"\n Recommendation Summary ")
print(f"-" * 50)
print(f" Model : {best_model_name}")
print(f" Data cutoff : {latest_date}")
print(f" Recommendation date : {today}")
print(f" Market : {market.upper()}")
print(f" Top N : {top_n}")
print(f"-" * 50)

display(recommendations.style.format({"Prediction Score": "{:.6f}"}).set_caption(f"Today's Stock Recommendations ({best_model_name})"))

In [ ]:
###################################
# Visualize top recommendations
###################################

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))
scores = top_stocks["score"].values
stocks = top_stocks.index.values
colors = ["#e74c3c" if i < 3 else "#3498db" for i in range(len(stocks))]

bars = ax.barh(range(len(stocks)), scores, color=colors)
ax.set_yticks(range(len(stocks)))
ax.set_yticklabels(stocks)
ax.invert_yaxis()
ax.set_xlabel("Prediction Score", fontsize=12)
ax.set_title(f"Top {top_n} Stock Recommendations ({today}) — {best_model_name}", fontsize=14, fontweight="bold")
ax.axvline(x=0, color="black", linewidth=0.5)

# Add score labels
for i, (score, stock) in enumerate(zip(scores, stocks)):
    ax.text(score + 0.001, i, f"{score:.4f}", va="center", fontsize=9)

ax.legend(
    [plt.Rectangle((0, 0), 1, 1, color="#e74c3c"), plt.Rectangle((0, 0), 1, 1, color="#3498db")],
    ["Top 3 (Strong Buy)", f"Top 4-{top_n} (Buy)"],
    loc="lower right",
)
plt.tight_layout()
plt.show()